<a href="https://colab.research.google.com/github/shehanidilmi52-ctrl/NorthStar-Analytics/blob/main/01_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
files <- system("ls /content/", intern=TRUE)
print(files)

[1] "sample_data"


In [ ]:
path <- '/content/'

drivers <- read.csv(paste0(path, 'drivers.csv'))
orders <- read.csv(paste0(path, 'orders.csv'))
deliveries <- read.csv(paste0(path, 'deliveries.csv'))
complaints <- read.csv(paste0(path, 'complaints.csv'))
vehicles <- read.csv(paste0(path, 'vehicles.csv'))
customers <- read.csv(paste0(path, 'customers.csv'))
incidents <- read.csv(paste0(path, 'incidents.csv'))
hubs <- read.csv(paste0(path, 'hubs.csv'))

cat("✅ All files loaded!\n")

Warning message in file(file, "rt"):
“cannot open file '/content/drivers.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
result1 <- sqldf("
  SELECT delivery_status,
         COUNT(*) as total,
         ROUND(AVG(fuel_or_charge_cost), 2) as avg_cost,
         ROUND(AVG(customer_rating_post_delivery), 2) as avg_rating
  FROM deliveries
  GROUP BY delivery_status
  ORDER BY total DESC
")
print(result1)

In [ ]:
result2 <- sqldf("
  SELECT o.pickup_zone,
         COUNT(*) as total_orders,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed,
         SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) as delayed
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY failed DESC
")
print(result2)

In [ ]:
result3 <- sqldf("
  SELECT d.driver_id,
         dr.driver_rating,
         dr.years_experience,
         COUNT(*) as total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failures,
         ROUND(AVG(d.manual_route_override_count), 2) as avg_overrides
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id
  ORDER BY failures DESC
  LIMIT 10
")
print(result3)

In [ ]:
result4 <- sqldf("
  SELECT complaint_type,
         severity,
         COUNT(*) as total,
         ROUND(AVG(resolution_days), 2) as avg_resolution_days,
         ROUND(AVG(compensation_amount), 2) as avg_compensation
  FROM complaints
  GROUP BY complaint_type, severity
  ORDER BY total DESC
")
print(result4)

In [ ]:
result5 <- sqldf("
  SELECT v.maintenance_status,
         COUNT(*) as total_deliveries,
         SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failures,
         ROUND(AVG(d.fuel_or_charge_cost), 2) as avg_cost
  FROM deliveries d
  JOIN vehicles v ON d.vehicle_id = v.vehicle_id
  GROUP BY v.maintenance_status
  ORDER BY failures DESC
")
print(result5)

In [ ]:
ggplot(result2, aes(x=reorder(pickup_zone, -failed), y=failed)) +
  geom_bar(stat='identity', fill='red') +
  labs(title='Failed Deliveries by Zone',
       x='Zone', y='Number of Failures') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle=45, hjust=1))

In [ ]:
ggplot(result1, aes(x=delivery_status, y=total, fill=delivery_status)) +
  geom_bar(stat='identity') +
  scale_fill_manual(values=c('OnTime'='green', 'Delayed'='orange', 'Failed'='red')) +
  labs(title='Delivery Status Summary',
       x='Status', y='Total Count') +
  theme_minimal()

In [ ]:
ggplot(result3, aes(x=reorder(driver_id, -failures), y=failures)) +
  geom_bar(stat='identity', fill='darkred') +
  labs(title='Top 10 Drivers by Failures',
       x='Driver ID', y='Number of Failures') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle=45, hjust=1))